# Définition des constantes

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, platform
from pathlib import Path
try:
    import torch
except ModuleNotFoundError:
    %pip install torch
    import torch
try:
    import pandas as pd
except ModuleNotFoundError:
    %pip install pandas
    import pandas as pd

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA = ROOT / "Data"
SRC = ROOT / "SRC"
SPLIT = DATA / "SPLIT"
SPLIT.mkdir(parents=True, exist_ok=True)


# Importation du jeu de données

In [ ]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv(DATA / "training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)
print(df.head())

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

df["TARGET_BINARY"] = df["TARGET"].map({0: 0, 4: 1})
dftrainval, dftest = train_test_split(df[["TARGET_BINARY", "tweet"]], test_size=50_000, random_state=42, stratify=df["TARGET_BINARY"])

dftrain, dfval = train_test_split(dftrainval, test_size=25_000, random_state=42, stratify=dftrainval["TARGET_BINARY"])


## Sauvegarde des splits

In [ ]:

#TRAIN
dftrain.to_csv(SPLIT / "train.csv", index=False, encoding="utf-8")

#VALIDATION
dfval.to_csv(SPLIT / "val.csv", index=False, encoding="utf-8")

#TEST
dftest.to_csv(SPLIT / "test.csv", index=False, encoding="utf-8")

# PIPELINE 1 (LSTM)

## Pré-traitement

### Nettoyage

In [4]:
from SRC.preprocessing import clean_tweet_LSTM

df_train_LSTM = pd.read_csv(SPLIT / "train.csv", encoding='utf-8')
df_train_LSTM = df_train_LSTM.sample(
    n=300_000, random_state=42
).reset_index(drop=True)


df_val_LSTM = pd.read_csv(SPLIT / "val.csv", encoding='utf-8')
df_test_LSTM = pd.read_csv(SPLIT / "test.csv", encoding='utf-8')

df_train_LSTM["tweet_net"] = df_train_LSTM["tweet"].apply(clean_tweet_LSTM)
df_val_LSTM["tweet_net"] = df_val_LSTM["tweet"].apply(clean_tweet_LSTM)
df_test_LSTM["tweet_net"] = df_test_LSTM["tweet"].apply(clean_tweet_LSTM)

print(df_train_LSTM[["tweet", "tweet_net"]].head())

                                               tweet  \
0  Watching Ramsay's Kitchen Nightmares now  Ew c...   
1  On my way too the beachh with thee bitchess  a...   
2  My hand is swollen, bruised and all  the shit ...   
3  @MrBillyBones although it would be the highlig...   
4             Going to the beach with Cody and Jake    

                                           tweet_net  
0  watching ramsays kitchen nightmares now ew coc...  
1  on my way too the beachh with thee bitchess ah...  
2  my hand is swollen bruised and all the shit hu...  
3  although it would be the highlight of the summ...  
4              going to the beach with cody and jake  


### Embedding (avec Glove)

In [5]:
import numpy as np
from collections import Counter

MAX_VOCAB = 30_000

# --- Vocabulaire à partir du train nettoyé ---
counter = Counter()
for txt in df_train_LSTM["tweet_net"]:
    counter.update(str(txt).split())

# index 0 = <pad>, index 1 = <unk>
itos = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(itos)}
print(f"Vocabulaire : {len(itos):,} tokens")

if not(Path(DATA / "Embedding" / "emb_matrix_300k.npy").exists()):

    EMB_DIM = 200
    GLOVE_PATH = DATA / "Embedding" / "glove.twitter.27B.200d.txt"


    # --- Matrice d'embeddings ---
    rng = np.random.default_rng(42)
    emb_matrix = rng.normal(0, 0.1, (len(itos), EMB_DIM)).astype(np.float32)
    emb_matrix[0] = 0.0  # <pad> à zéro

    found = 0
    with open(GLOVE_PATH, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            idx = stoi.get(word)
            if idx is not None:
                emb_matrix[idx] = np.asarray(parts[1:], dtype=np.float32)
                found += 1

    print(f"Couverture GloVe : {found:,}/{len(itos):,} ({found/len(itos):.1%})")

    np.save(DATA / "Embedding" / "emb_matrix_300k.npy", emb_matrix)
    lengths = df_train_LSTM["tweet_net"].str.split().str.len()
    print(lengths.describe())
    print(f"p95 : {lengths.quantile(0.95):.0f} | p99 : {lengths.quantile(0.99):.0f}")
else:
    emb_matrix = np.load(DATA / "Embedding" / "emb_matrix_300k.npy")

Vocabulaire : 30,000 tokens


## Entrainement

### tweets => Matrices

In [6]:
import numpy as np
import torch

MAX_LEN = 40
PAD, UNK = 0, 1

def textes_vers_matrice(series_textes):
    """Convertit une colonne de textes en matrice (n_tweets, 40)."""
    matrice = np.zeros((len(series_textes), MAX_LEN), dtype=np.int64)
    for i, texte in enumerate(series_textes):
        mots = str(texte).split()[:MAX_LEN]
        for j, mot in enumerate(mots):
            matrice[i, j] = stoi.get(mot, UNK)
    return matrice

# Conversion en tenseurs
X_train = torch.tensor(textes_vers_matrice(df_train_LSTM["tweet_net"]))
y_train = torch.tensor(df_train_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

X_val = torch.tensor(textes_vers_matrice(df_val_LSTM["tweet_net"]))
y_val = torch.tensor(df_val_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

X_test = torch.tensor(textes_vers_matrice(df_test_LSTM["tweet_net"]))
y_test = torch.tensor(df_test_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

print(X_train.shape, y_train.shape)

torch.Size([300000, 40]) torch.Size([300000])


### Modèle

In [7]:
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

class LSTMSentiment(nn.Module):
    def __init__(self, emb_matrix):
        super().__init__()
        # 1. Embedding : indice -> vecteur GloVe de dim 200
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix), freeze=False, padding_idx=PAD
        )
        # 2. LSTM bidirectionnel : lit le tweet dans les deux sens
        self.lstm = nn.LSTM(200, 128, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        # 3. Sortie : 256 -> 1 score
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        emb = self.embedding(x)              # (batch, 40, 200)
        sorties, _ = self.lstm(emb)          # (batch, 40, 256)
        moyenne = sorties.mean(dim=1)        # (batch, 256)
        return self.fc(self.dropout(moyenne)).squeeze(1)

model = LSTMSentiment(emb_matrix).to(DEVICE)

cpu


### boucle d'entrainement

In [8]:
from sklearn.metrics import accuracy_score, f1_score
import time, copy

best_f1, best_state, patience = 0, None, 0
BATCH = 256
EPOCHS = 5

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def predire(X):
    model.eval()
    predictions = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            logits = model(X[i:i+512].to(DEVICE))
            predictions += (torch.sigmoid(logits) > 0.5).long().cpu().tolist()
    return predictions


for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(X_train))     # mélange à chaque epoch
    perte_totale, t0 = 0, time.time()

    for i in range(0, len(X_train), BATCH):
        idx = perm[i:i+BATCH]
        xb, yb = X_train[idx].to(DEVICE), y_train[idx].to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        perte_totale += loss.item()
    preds = predire(X_val)
    f1 = f1_score(y_val, preds, average="macro")
    acc = accuracy_score(y_val, preds)
    print(f"Epoch {epoch} | loss {perte_totale/(len(X_train)//BATCH):.4f} | "
          f"val acc {acc:.4f} | val F1 {f1:.4f} | {time.time()-t0:.0f}s")

    if f1 > best_f1:
        best_f1, best_state, patience = f1, copy.deepcopy(model.state_dict()), 0
    else:
        patience += 1
        if patience >= 2:
            print(f"Early stopping — meilleur F1 val : {best_f1:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modèle restauré — F1 val : {best_f1:.4f}")

Epoch 1 | loss 0.4485 | val acc 0.8090 | val F1 0.8090 | 139s
Epoch 2 | loss 0.3897 | val acc 0.8190 | val F1 0.8190 | 150s
Epoch 3 | loss 0.3533 | val acc 0.8184 | val F1 0.8184 | 151s
Epoch 4 | loss 0.3171 | val acc 0.8158 | val F1 0.8158 | 152s
Early stopping — meilleur F1 val : 0.8190
Modèle restauré — F1 val : 0.8190


## Tests

In [9]:
# Sauvegarde du meilleur modèle LSTM AVANT les tests
from pathlib import Path
import torch

MODELS_DIR = ROOT / "Models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

LSTM_PATH = MODELS_DIR / "lstm_baseline.pt"

model.eval()

torch.save(
    {
        "state_dict": model.state_dict(),
        "itos": itos,
        "stoi": stoi,
        "max_len": MAX_LEN,
        "pad_index": PAD,
        "unk_index": UNK,
        "embedding_dim": 200,
        "hidden_dim": 128,
        "bidirectional": True,
        "threshold": 0.5,
        "best_f1_validation": float(best_f1)
    },
    LSTM_PATH
)

print(f"✅ LSTM sauvegardé dans : {LSTM_PATH}")

✅ LSTM sauvegardé dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Models\lstm_baseline.pt


In [10]:
from sklearn.metrics import confusion_matrix, classification_report

preds_test = predire(X_test)

print(confusion_matrix(y_test, preds_test))
print(classification_report(y_test, preds_test, target_names=["négatif", "positif"]))

[[20673  4327]
 [ 4947 20053]]
              precision    recall  f1-score   support

     négatif       0.81      0.83      0.82     25000
     positif       0.82      0.80      0.81     25000

    accuracy                           0.81     50000
   macro avg       0.81      0.81      0.81     50000
weighted avg       0.81      0.81      0.81     50000



# PIPELINE 2 (SETFIT)

## Pré-traitement

### Nettoyage

In [11]:
from SRC.preprocessing import clean_tweet_SETFIT
from datasets import Dataset

df_train_SETFIT = pd.read_csv(SPLIT / "train.csv", encoding='utf-8')


# Rechargement des splits (nettoyage SetFit déjà appliqué au train)
df_val_SETFIT = pd.read_csv(SPLIT / "val.csv", encoding="utf-8")
df_test_SETFIT = pd.read_csv(SPLIT / "test.csv", encoding="utf-8")

df_val_SETFIT["tweet_net"] = df_val_SETFIT["tweet"].apply(clean_tweet_SETFIT)
df_test_SETFIT["tweet_net"] = df_test_SETFIT["tweet"].apply(clean_tweet_SETFIT)

# --- Échantillonnage few-shot : N exemples par classe ---
N_PER_CLASS = 512

df_fewshot = (
    df_train_SETFIT
    .groupby("TARGET_BINARY")
    .sample(n=N_PER_CLASS, random_state=42)
    .reset_index(drop=True)
)
df_fewshot["tweet_net"] = df_fewshot["tweet"].apply(clean_tweet_SETFIT)

df_val_small = (
    df_val_SETFIT
    .groupby("TARGET_BINARY", group_keys=False)
    .sample(n=1_000, random_state=42)
    .reset_index(drop=True)
)

print(df_val_small["TARGET_BINARY"].value_counts())

print(f"Train few-shot : {len(df_fewshot)} exemples "
      f"({N_PER_CLASS} par classe)")
print(f"Validation     : {len(df_val_small)}")
print(f"Test           : {len(df_test_SETFIT)}")

# Conversion au format HuggingFace Dataset
train_ds = Dataset.from_pandas(
    df_fewshot[["tweet_net", "TARGET_BINARY"]]
    .rename(columns={"tweet_net": "text", "TARGET_BINARY": "label"})
)
val_ds = Dataset.from_pandas(
    df_val_small[["tweet_net", "TARGET_BINARY"]]
    .rename(columns={"tweet_net": "text", "TARGET_BINARY": "label"})
)



f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TARGET_BINARY
0    1000
1    1000
Name: count, dtype: int64
Train few-shot : 1024 exemples (512 par classe)
Validation     : 2000
Test           : 50000


## Entrainement

In [ ]:
"""rom setfit import SetFitModel, Trainer, TrainingArguments
import time

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model_setfit = SetFitModel.from_pretrained(
    MODEL_NAME,
    labels=["négatif", "positif"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=(2, 16),
    num_iterations=20,
    body_learning_rate=2e-5,
    head_learning_rate=1e-2,
    max_length=64,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model_setfit,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    metric="accuracy",
)

t0 = time.time()
trainer.train()
print(f"Entraînement terminé en {time.time() - t0:.0f}s")

metrics = trainer.evaluate()
print(metrics)"""
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import time

MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"

model_setfit = SetFitModel.from_pretrained(
    MODEL_NAME,
    labels=["négatif", "positif"]
)

def compute_metrics(y_pred, y_true):
    y_pred = np.asarray(y_pred)
    y_true = np.asarray(y_true)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro")
    }

args = TrainingArguments(
    output_dir=str(ROOT / "Models" / "checkpoints_setfit"),
    batch_size=(16, 16),
    num_epochs=(1, 8),
    num_iterations=10,
    body_learning_rate=2e-5,
    head_learning_rate=1e-2,
    max_length=64,
    seed=42,
    report_to="none",
    save_strategy="no"
)

trainer = Trainer(
    model=model_setfit,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    metric=compute_metrics
)


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Map: 100%|██████████| 1024/1024 [00:00<00:00, 44511.58 examples/s]
***** Running training *****
  Num unique pairs = 20480
  Batch size = 16
  Num epochs = 1
f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.438100
50,0.282000
100,0.236000
150,0.191600
200,0.149700
250,0.109200
300,0.082500
350,0.074100
400,0.055300
450,0.037500


***** Running evaluation *****


Entraînement terminé en 1799.0 secondes
{'accuracy': 0.823, 'f1_macro': 0.8229936277705998}


In [14]:
import json
import time

t0 = time.time()
trainer.train()
training_time_setfit = time.time() - t0

print(f"Entraînement terminé en {training_time_setfit:.1f} secondes")

# Sauvegarde immédiate AVANT l'évaluation et les tests
SETFIT_DIR = ROOT / "Models" / "setfit_mpnet_best"
SETFIT_DIR.mkdir(parents=True, exist_ok=True)

model_setfit.save_pretrained(str(SETFIT_DIR))

training_info = {
    "model_name": MODEL_NAME,
    "training_time_seconds": training_time_setfit,
    "training_examples": len(train_ds),
    "examples_per_class": N_PER_CLASS
}

with open(
    SETFIT_DIR / "training_info.json",
    "w",
    encoding="utf-8"
) as fichier:
    json.dump(training_info, fichier, indent=4, ensure_ascii=False)

print(f"✅ SetFit sauvegardé dans : {SETFIT_DIR}")

***** Running training *****
  Num unique pairs = 20480
  Batch size = 16
  Num epochs = 1
f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.000300
50,0.006400
100,0.002500
150,0.001800
200,0.004100
250,0.009700
300,0.005800
350,0.010600
400,0.005300
450,0.001300


Entraînement terminé en 1784.0 secondes
✅ SetFit sauvegardé dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Models\setfit_mpnet_best


In [ ]:
import json

# Évaluation
metrics_setfit = trainer.evaluate()

# Dossier de sauvegarde
SETFIT_DIR = ROOT / "Models" / "setfit_mpnet_best"
SETFIT_DIR.mkdir(parents=True, exist_ok=True)

# Sauvegarde complète du modèle SetFit
model_setfit.save_pretrained(str(SETFIT_DIR))

# Sauvegarde des résultats
informations = {
    "model_name": MODEL_NAME,
    "training_time_seconds": training_time_setfit,
    "validation_accuracy": float(metrics_setfit["accuracy"]),
    "validation_f1_macro": float(metrics_setfit["f1_macro"]),
    "training_examples": len(train_ds)
}

with open(SETFIT_DIR / "metrics.json", "w", encoding="utf-8") as fichier:
    json.dump(informations, fichier, indent=4, ensure_ascii=False)

print(f"Modèle sauvegardé dans : {SETFIT_DIR}")
print(informations)

In [ ]:
texts_val = df_val_small["tweet_net"].tolist()
y_val_setfit = df_val_small["TARGET_BINARY"].to_numpy()

preds_val_setfit = np.asarray(
    model_setfit.predict(texts_val)
)

if preds_val_setfit.dtype.kind in "OU":
    preds_val_setfit = np.where(
        preds_val_setfit == "positif", 1, 0
    )

val_accuracy = accuracy_score(y_val_setfit, preds_val_setfit)
val_f1 = f1_score(y_val_setfit, preds_val_setfit, average="macro")

print(f"Accuracy validation : {val_accuracy:.4f}")
print(f"F1 macro validation : {val_f1:.4f}")

In [ ]:
N_VALUES = [64, 128, 256, 512, 1000]
resultats_validation = []

for n_per_class in N_VALUES:

    train_sample = (
        df_train_SETFIT
        .groupby("TARGET_BINARY", group_keys=False)
        .sample(n=n_per_class, random_state=42)
        .reset_index(drop=True)
    )

    train_sample["tweet_net"] = (
        train_sample["tweet"]
        .fillna("")
        .apply(clean_tweet_SETFIT)
    )

    train_ds_exp = Dataset.from_pandas(
        train_sample[["tweet_net", "TARGET_BINARY"]]
        .rename(columns={
            "tweet_net": "text",
            "TARGET_BINARY": "label"
        }),
        preserve_index=False
    )

    model_exp = SetFitModel.from_pretrained(
        MODEL_NAME,
        labels=["négatif", "positif"]
    )

    trainer_exp = Trainer(
        model=model_exp,
        args=args,
        train_dataset=train_ds_exp,
        eval_dataset=val_ds,
        metric=compute_metrics
    )

    start = time.time()
    trainer_exp.train()
    duration = time.time() - start

    preds = np.asarray(
        model_exp.predict(df_val_small["tweet_net"].tolist())
    )

    if preds.dtype.kind in "OU":
        preds = np.where(preds == "positif", 1, 0)

    resultats_validation.append({
        "Exemples par classe": n_per_class,
        "Exemples totaux": 2 * n_per_class,
        "Accuracy validation": accuracy_score(y_val_setfit, preds),
        "F1 macro validation": f1_score(
            y_val_setfit, preds, average="macro"
        ),
        "Temps entraînement": duration
    })

df_resultats_validation = pd.DataFrame(resultats_validation)
display(df_resultats_validation)

# Tests

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report
)
import numpy as np

def predire_setfit(textes, batch=256):
    """Prédiction par lots pour éviter la saturation mémoire."""
    preds = []
    for i in range(0, len(textes), batch):
        sortie = model_setfit.predict(list(textes[i:i+batch]))
        preds.extend(sortie)
    return np.array(preds)

textes_test = df_test_SETFIT["tweet_net"].tolist()
y_test_setfit = df_test_SETFIT["TARGET_BINARY"].values

t0 = time.time()
preds_setfit = predire_setfit(textes_test)
print(f"Inférence sur {len(textes_test):,} tweets : {time.time() - t0:.0f}s")

# Les labels reviennent en chaînes -> remise en binaire
if preds_setfit.dtype.kind in "OU":
    preds_setfit = np.where(preds_setfit == "positif", 1, 0)

print(confusion_matrix(y_test_setfit, preds_setfit))
print(classification_report(
    y_test_setfit, preds_setfit,
    target_names=["négatif", "positif"]
))

# Comparaison finale des modèles sur le jeu de test

In [ ]:
import numpy as np
import pandas as pd
import time

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

assert len(df_test_LSTM) == len(df_test_SETFIT)
assert np.array_equal(
    df_test_LSTM["TARGET_BINARY"].to_numpy(),
    df_test_SETFIT["TARGET_BINARY"].to_numpy()
)

y_test_final = df_test_LSTM["TARGET_BINARY"].to_numpy()

print(f"Jeu de test commun : {len(y_test_final):,} tweets")
print(pd.Series(y_test_final).value_counts())

In [ ]:
model.eval()

start = time.time()
predictions_lstm_final = np.asarray(predire(X_test), dtype=int)
temps_inference_lstm = time.time() - start

metrics_lstm_test = {
    "accuracy": accuracy_score(
        y_test_final,
        predictions_lstm_final
    ),
    "f1_macro": f1_score(
        y_test_final,
        predictions_lstm_final,
        average="macro"
    ),
    "temps_inference": temps_inference_lstm
}

print("LSTM — résultats sur le test")
print(metrics_lstm_test)

print(
    classification_report(
        y_test_final,
        predictions_lstm_final,
        target_names=["négatif", "positif"]
    )
)

In [ ]:
def convertir_predictions_setfit(predictions):
    predictions = np.asarray(predictions)

    # SetFit peut retourner des labels textuels
    if predictions.dtype.kind in "OUS":
        mapping = {
            "négatif": 0,
            "positif": 1,
            "negative": 0,
            "positive": 1,
            "0": 0,
            "1": 1
        }

        predictions = np.array(
            [mapping.get(str(prediction), prediction)
             for prediction in predictions],
            dtype=int
        )

    return predictions.astype(int)


def predire_setfit_par_lots(model, textes, batch_size=256):
    predictions = []

    for debut in range(0, len(textes), batch_size):
        lot = textes[debut:debut + batch_size]
        predictions.extend(model.predict(lot))

    return convertir_predictions_setfit(predictions)

In [ ]:
textes_test_setfit = (
    df_test_SETFIT["tweet"]
    .fillna("")
    .apply(clean_tweet_SETFIT)
    .tolist()
)

start = time.time()

predictions_setfit_final = predire_setfit_par_lots(
    model=model_setfit,
    textes=textes_test_setfit,
    batch_size=256
)

temps_inference_setfit = time.time() - start

metrics_setfit_test = {
    "accuracy": accuracy_score(
        y_test_final,
        predictions_setfit_final
    ),
    "f1_macro": f1_score(
        y_test_final,
        predictions_setfit_final,
        average="macro"
    ),
    "temps_inference": temps_inference_setfit
}

print("SetFit — résultats sur le test")
print(metrics_setfit_test)

print(
    classification_report(
        y_test_final,
        predictions_setfit_final,
        target_names=["négatif", "positif"]
    )
)

In [ ]:
resultats_test = pd.DataFrame([
    {
        "Modèle": "BiLSTM + GloVe",
        "Type": "Baseline",
        "Backbone": "GloVe Twitter 200d",
        "Exemples entraînement": 300_000,
        "Accuracy test": metrics_lstm_test["accuracy"],
        "F1 macro test": metrics_lstm_test["f1_macro"],
        "Temps inférence (s)": metrics_lstm_test["temps_inference"]
    },
    {
        "Modèle": "SetFit + MPNet",
        "Type": "Méthode récente",
        "Backbone": "paraphrase-mpnet-base-v2",
        "Exemples entraînement": len(df_fewshot),
        "Accuracy test": metrics_setfit_test["accuracy"],
        "F1 macro test": metrics_setfit_test["f1_macro"],
        "Temps inférence (s)": metrics_setfit_test["temps_inference"]
    }
])

display(
    resultats_test.style.format({
        "Accuracy test": "{:.4f}",
        "F1 macro test": "{:.4f}",
        "Temps inférence (s)": "{:.1f}"
    })
)

resultats_test.to_csv(
    ROOT / "Models" / "comparaison_finale_test.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

matrices = [
    ("BiLSTM + GloVe", predictions_lstm_final),
    ("SetFit + MPNet", predictions_setfit_final)
]

for axe, (titre, predictions) in zip(axes, matrices):
    matrice = confusion_matrix(y_test_final, predictions)

    sns.heatmap(
        matrice,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axe,
        xticklabels=["négatif", "positif"],
        yticklabels=["négatif", "positif"]
    )

    axe.set_title(titre)
    axe.set_xlabel("Classe prédite")
    axe.set_ylabel("Classe réelle")

plt.tight_layout()
plt.savefig(
    ROOT / "Models" / "matrices_confusion_test.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
resultats_long = resultats_test.melt(
    id_vars="Modèle",
    value_vars=["Accuracy test", "F1 macro test"],
    var_name="Métrique",
    value_name="Score"
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=resultats_long,
    x="Modèle",
    y="Score",
    hue="Métrique"
)

plt.ylim(0.70, 0.86)
plt.ylabel("Score sur le jeu de test")
plt.title("Comparaison finale sur le même jeu de test")

plt.axhline(0.8, color="grey", linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()

plt.savefig(
    ROOT / "Models" / "comparaison_modeles_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Sauvegarde

In [ ]:
import pandas as pd

resultats = pd.DataFrame([
    {
        "Modèle": "LSTM + GloVe (baseline)",
        "Exemples d'entraînement": 300_000,
        "Accuracy": accuracy_score(y_test, preds_test),
        "F1 macro": f1_score(y_test, preds_test, average="macro"),
    },
    {
        "Modèle": f"SetFit ({N_PER_CLASS}/classe)",
        "Exemples d'entraînement": len(df_fewshot),
        "Accuracy": accuracy_score(y_test_setfit, preds_setfit),
        "F1 macro": f1_score(y_test_setfit, preds_setfit, average="macro"),
    },
])

resultats["Accuracy"] = resultats["Accuracy"].round(4)
resultats["F1 macro"] = resultats["F1 macro"].round(4)

print(resultats.to_string(index=False))
resultats.to_csv(ROOT / "Models" / "comparaison.csv", index=False)

# Comparaison finale des modèles sur le jeu de test

## Rechargement des modèles sauvegardés

In [6]:
import torch
import torch.nn as nn
import numpy as np

# Choix automatique : GPU s'il est disponible, sinon CPU
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

class LSTMSentiment(nn.Module):
    def __init__(self, emb_matrix):
        super().__init__()
        # 1. Embedding : indice -> vecteur GloVe de dim 200
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix), freeze=False, padding_idx=PAD
        )
        # 2. LSTM bidirectionnel : lit le tweet dans les deux sens
        self.lstm = nn.LSTM(200, 128, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        # 3. Sortie : 256 -> 1 score
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        emb = self.embedding(x)              # (batch, 40, 200)
        sorties, _ = self.lstm(emb)          # (batch, 40, 256)
        moyenne = sorties.mean(dim=1)        # (batch, 256)
        return self.fc(self.dropout(moyenne)).squeeze(1)

print(f"Appareil utilisé : {DEVICE}")

LSTM_PATH = ROOT / "Models" / "lstm_baseline.pt"

checkpoint_lstm = torch.load(
    LSTM_PATH,
    map_location=DEVICE,
    weights_only=False
)

# Récupération du vocabulaire
itos = checkpoint_lstm["itos"]
stoi = checkpoint_lstm["stoi"]

MAX_LEN = checkpoint_lstm["max_len"]
PAD = checkpoint_lstm["pad_index"]
UNK = checkpoint_lstm["unk_index"]

# Les embeddings entraînés sont présents dans le state_dict
embedding_weights = (
    checkpoint_lstm["state_dict"]["embedding.weight"]
    .cpu()
    .numpy()
)

model = LSTMSentiment(embedding_weights).to(DEVICE)
model.load_state_dict(checkpoint_lstm["state_dict"])
model.eval()

print("✅ Modèle LSTM rechargé")
print(f"F1 validation sauvegardé : {checkpoint_lstm['best_f1_validation']:.4f}")

Appareil utilisé : cpu
✅ Modèle LSTM rechargé
F1 validation sauvegardé : 0.8190


In [7]:
from setfit import SetFitModel

SETFIT_DIR = ROOT / "Models" / "setfit_mpnet_best"

model_setfit = SetFitModel.from_pretrained(str(SETFIT_DIR))

print("✅ Modèle SetFit rechargé")

f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Modèle SetFit rechargé


In [ ]:
import time
import numpy as np
import pandas as pd
import torch


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. Préparation du même jeu de test pour les deux modèles
# ============================================================

TEXT_COLUMN = "tweet"
dftest = pd.read_csv(SPLIT / "test.csv", encoding='utf-8')

texts_test = dftest[TEXT_COLUMN].fillna("").astype(str).tolist()
y_true = dftest["TARGET_BINARY"].astype(int).to_numpy()

print(f"Nombre d'exemples testés : {len(texts_test):,}")
print(f"Appareil utilisé : {DEVICE}")


# ============================================================
# 2. Encodage des textes pour le LSTM
# ============================================================

def encoder_textes_lstm(textes):
    textes_encodes = []

    for texte in textes:
        tokens = texte.split()

        indices = [
            stoi.get(token, UNK)
            for token in tokens[:MAX_LEN]
        ]

        # Padding si le texte est trop court
        indices += [PAD] * (MAX_LEN - len(indices))
        textes_encodes.append(indices)

    return torch.tensor(textes_encodes, dtype=torch.long)


X_test_lstm = encoder_textes_lstm(texts_test)


# ============================================================
# 3. Prédictions du LSTM
# ============================================================

model.eval()
predictions_lstm = []

debut = time.perf_counter()

with torch.no_grad():
    for i in range(0, len(X_test_lstm), 512):
        batch = X_test_lstm[i:i + 512].to(DEVICE)

        logits = model(batch)
        probabilites = torch.sigmoid(logits)

        predictions = (probabilites >= 0.5).long()
        predictions_lstm.extend(predictions.cpu().numpy())

temps_lstm = time.perf_counter() - debut
predictions_lstm = np.asarray(predictions_lstm, dtype=int)


# ============================================================
# 4. Prédictions de SetFit
# ============================================================

debut = time.perf_counter()

predictions_setfit = np.asarray(
    model_setfit.predict(texts_test)
)

temps_setfit = time.perf_counter() - debut

# Conversion éventuelle des labels textuels
if predictions_setfit.dtype.kind in "OUS":
    predictions_setfit = np.array([
        1 if str(label).lower() in ["1", "positif", "positive"]
        else 0
        for label in predictions_setfit
    ])

predictions_setfit = predictions_setfit.astype(int)


# ============================================================
# 5. Calcul des métriques
# ============================================================

resultats_test = pd.DataFrame([
    {
        "Modèle": "BiLSTM + GloVe",
        "Accuracy": accuracy_score(y_true, predictions_lstm),
        "F1 macro": f1_score(
            y_true,
            predictions_lstm,
            average="macro"
        ),
        "Temps d'inférence (s)": temps_lstm
    },
    {
        "Modèle": "SetFit + MPNet",
        "Accuracy": accuracy_score(y_true, predictions_setfit),
        "F1 macro": f1_score(
            y_true,
            predictions_setfit,
            average="macro"
        ),
        "Temps d'inférence (s)": temps_setfit
    }
])

display(
    resultats_test.style
    .format({
        "Accuracy": "{:.4f}",
        "F1 macro": "{:.4f}",
        "Temps d'inférence (s)": "{:.1f}"
    })
    .highlight_max(
        subset=["Accuracy", "F1 macro"],
        color="lightgreen"
    )
    .highlight_min(
        subset=["Temps d'inférence (s)"],
        color="lightgreen"
    )
)


# ============================================================
# 6. Rapports détaillés
# ============================================================

print("\n===== BiLSTM + GloVe =====")
print(confusion_matrix(y_true, predictions_lstm))
print(classification_report(
    y_true,
    predictions_lstm,
    target_names=["négatif", "positif"],
    digits=4
))

print("\n===== SetFit + MPNet =====")
print(confusion_matrix(y_true, predictions_setfit))
print(classification_report(
    y_true,
    predictions_setfit,
    target_names=["négatif", "positif"],
    digits=4
))


# ============================================================
# 7. Sauvegarde des résultats
# ============================================================

RESULTATS_PATH = ROOT / "Models" / "comparaison_modeles_recharges.csv"

resultats_test.to_csv(
    RESULTATS_PATH,
    index=False
)

print(f"\n✅ Résultats sauvegardés dans : {RESULTATS_PATH}")

Nombre d'exemples testés : 50,000
Appareil utilisé : cpu


,Modèle,Accuracy,F1 macro,Temps d'inférence (s)
0,BiLSTM + GloVe,0.7576,0.7570,2.6
1,SetFit + MPNet,0.8185,0.8184,381.8



===== BiLSTM + GloVe =====
[[17629  7371]
 [ 4747 20253]]
              precision    recall  f1-score   support

     négatif     0.7879    0.7052    0.7442     25000
     positif     0.7332    0.8101    0.7697     25000

    accuracy                         0.7576     50000
   macro avg     0.7605    0.7576    0.7570     50000
weighted avg     0.7605    0.7576    0.7570     50000


===== SetFit + MPNet =====
[[20048  4952]
 [ 4124 20876]]
              precision    recall  f1-score   support

     négatif     0.8294    0.8019    0.8154     25000
     positif     0.8083    0.8350    0.8214     25000

    accuracy                         0.8185     50000
   macro avg     0.8188    0.8185    0.8184     50000
weighted avg     0.8188    0.8185    0.8184     50000


✅ Résultats sauvegardés dans : f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Models\comparaison_modeles_recharges.csv
